# Experiment 2 — salting the skewed join

Same join as Experiment 1, same settings (AQE skew handling off). You implement the salting; the notebook checks the result.

In [1]:
import time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m5-exp2-salting").getOrCreate()
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")

fact = spark.read.parquet("/opt/data/m5/orders_skew")
dim = spark.read.parquet("/opt/data/m4/customers_big")

HOT_KEYS = [1]
N_SALT = 16

def timed(label, df):
    t0 = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.perf_counter() - t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 19:24:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Your implementation

Build `salted`, a DataFrame with the same rows as `fact.join(dim, "customer_id")`, using a salt column so that the hot key's rows are spread over `N_SALT` partitions.

Steps to cover:
1. On `fact`, add a `salt` column. Decide what it is for hot keys and for all other keys.
2. On `dim`, produce the matching `salt` values: how many rows must each `dim` row become?
3. Join on `customer_id` **and** `salt`, then drop `salt`.

**PREDICT** before running it: the max task duration of the join stage relative to Experiment 1; the shuffle write of each side relative to Experiment 1; the runtime.

In [7]:
is_hot = F.col("customer_id").isin(HOT_KEYS)

fact_s = fact.withColumn(
    "salt",
    F.when(is_hot, F.pmod(F.hash("order_id"), F.lit(N_SALT)))   # hot key: spread over 0..15
     .otherwise(F.lit(0)),                                       # everyone else: one value
)

salts_all = F.array([F.lit(i) for i in range(N_SALT)])   # [0, 1, …, 15]
salts_one = F.array(F.lit(0))                             # [0]

dim_s = dim.withColumn("salt", F.explode(F.when(is_hot, salts_all).otherwise(salts_one)))


salted =  fact_s.join(dim_s, ["customer_id", "salt"]).drop('salt')

In [8]:
salted.explain("formatted")
timed("salted join", salted)

== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- SortMergeJoin Inner (11)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Scan parquet  (1)
      +- Sort (10)
         +- Exchange (9)
            +- Generate (8)
               +- Filter (7)
                  +- Scan parquet  (6)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m5/orders_skew]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) Filter
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : (isnotnull(customer_id#1) AND CASE WHEN (customer_id#1 = 1) THEN isnotnull(pmod(hash(order_id#0L, 42), 16)) ELSE true END)

(3) Project
Output [7]: [order_id#0L, customer_id#1, shop_id#2, amount#3, ord

[Stage 6:=============================>                            (8 + 8) / 16]

salted join: 5.9s


## Correctness check

The salted join must produce exactly the rows of the plain join: same total, same count for the hot key, same columns.

In [9]:
plain = fact.join(dim, "customer_id")
hot = F.col("customer_id").isin(HOT_KEYS)
print("columns equal:", sorted(plain.columns) == sorted(salted.columns))
print("hot-key rows  plain vs salted:", plain.filter(hot).count(), salted.filter(hot).count())
print("other rows    plain vs salted:", plain.filter(~hot).count(), salted.filter(~hot).count())

columns equal: True


hot-key rows  plain vs salted: 6000000 6000000
other rows    plain vs salted: 8000000 8000000


## Stop before opening the next notebook

In [ ]:
spark.stop()

In [10]:
timed("salted (warm)", salted)
timed("plain  (warm)", plain)

salted (warm): 2.8s


[Stage 52:===================================================>    (11 + 1) / 12]

plain  (warm): 4.4s
